In [1]:
#0
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
#1
# ===== SINGLE INITIALIZATION CELL (RUN ONCE) =====

# Imports
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from collections import defaultdict
from scipy.signal import butter, filtfilt


print("Imports loaded")

# Dataset class
class EEGDataset(Dataset):
    def __init__(self, X, y, trial_ids=None):
        self.X = X              # keep as numpy
        self.y = y
        self.trial_ids = trial_ids

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx]).float()
        y = torch.tensor(self.y[idx]).long()

        if self.trial_ids is not None:
            return x, y, self.trial_ids[idx]
        return x, y
# ================= Frequency Decomposition =================

def bandpass(x, low, high, fs=128, order=4):
    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, x, axis=-1)

def get_frequency_views(x):
    return {
        "delta": bandpass(x, 1, 4),
        "theta": bandpass(x, 4, 8),
        "alpha": bandpass(x, 8, 13),
        "beta":  bandpass(x, 13, 30)
    }





Imports loaded


In [4]:

#2
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/EEG_TSSL_DATA"

windows = np.load(os.path.join(DATA_PATH, "windows.npy"))
labels = np.load(os.path.join(DATA_PATH, "labels_windows.npy"))
trial_ids = np.load(os.path.join(DATA_PATH, "trial_ids.npy"))
bandpower_features = np.load(os.path.join(DATA_PATH, "bandpower_features.npy"))
print("Data loaded:", windows.shape, labels.shape, trial_ids.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data loaded: (78080, 32, 256) (78080,) (78080,)


In [5]:
#3
unique_trials = np.unique(trial_ids)
trial_labels = np.array([labels[trial_ids == t][0] for t in unique_trials])

#Split at the trial level
train_trials, test_trials = train_test_split(
    unique_trials,
    test_size=0.3,
    random_state=42,
    stratify=trial_labels
)

#Leakage check
print("Overlap:", len(set(train_trials) & set(test_trials)))


Overlap: 0


In [7]:
#4
train_mask = np.isin(trial_ids, train_trials)
test_mask  = np.isin(trial_ids, test_trials)

x_train = bandpower_features[train_mask]
X_test  = bandpower_features[test_mask]

y_train = labels[train_mask]
y_test  = labels[test_mask]
trial_id_test = trial_ids[test_mask]

print(X_train.shape, X_test.shape)
train_loader = DataLoader(
    EEGDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    EEGDataset(X_test, y_test, trial_id_test),
    batch_size=16,
    shuffle=False
)

print("✅ DataLoaders created successfully")



(54656, 128) (23424, 128)
✅ DataLoaders created successfully


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

clf = RandomForestClassifier(n_estimators=300, random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)

print("Window accuracy:", accuracy_score(y_test, preds))

Window accuracy: 0.5522540983606558


In [9]:
#5
train_loader = DataLoader(
    EEGDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    EEGDataset(X_test, y_test, trial_id_test),
    batch_size=64,
    shuffle=False
)

print("✅ DataLoaders created successfully")


✅ DataLoaders created successfully


In [23]:
#6

class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(32, 64, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(2)

        # 256 → 128 → 64 after two pools
        self.fc1 = nn.Linear(128 * 64, 128)
        self.fc2 = nn.Linear(128, 4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


In [24]:
EPOCHS = 8
lambda_freq = 0.5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        X = batch[0].to(device)
        y = batch[1].to(device)

        # -------- Frequency views --------
        X_np = X.detach().cpu().numpy()
        freq_views = get_frequency_views(X_np)

        X_alpha = torch.from_numpy(freq_views["alpha"].copy()).float().to(device)
        X_beta  = torch.from_numpy(freq_views["beta"].copy()).float().to(device)

        optimizer.zero_grad()

        # -------- Forward passes --------
        z_time  = model(X)
        z_alpha = model(X_alpha)
        z_beta  = model(X_beta)

        # -------- Normalize --------
        z_alpha = torch.nn.functional.normalize(z_alpha, dim=1)
        z_beta  = torch.nn.functional.normalize(z_beta, dim=1)

        L_freq = 1 - torch.mean(torch.sum(z_alpha * z_beta, dim=1))

        loss = criterion(z_time, y) + lambda_freq * L_freq

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

RuntimeError: Given groups=1, weight of size [64, 32, 7], expected input[1, 64, 128] to have 32 channels, but got 64 channels instead

In [11]:
#7
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BaselineCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Device:", device)


Device: cuda


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for X, y, _ in test_loader:
        X = X.to(device)
        outputs = model(X)
        preds = outputs.argmax(dim=1).cpu().numpy()

        y_true.extend(y.numpy())
        y_pred.extend(preds)

print("Window Accuracy:", accuracy_score(y_true, y_pred))
print("Window F1-score:", f1_score(y_true, y_pred, average='weighted'))

Window Accuracy: 0.4678107923497268
Window F1-score: 0.474314134603905


In [ ]:
from collections import defaultdict
from sklearn.metrics import confusion_matrix

trial_preds = defaultdict(list)
trial_true = {}

model.eval()
with torch.no_grad():
    for X, y, tids in test_loader:
        X = X.to(device)
        outputs = model(X)
        preds = outputs.argmax(dim=1).cpu().numpy()

        for i in range(len(preds)):
            tid = tids[i]
            trial_preds[tid].append(preds[i])
            trial_true[tid] = y[i].item()


In [ ]:
from collections import defaultdict
from sklearn.metrics import confusion_matrix

trial_test_ids = trial_ids[test_mask]

trial_preds = defaultdict(list)
trial_true = {}

for p, y, tid in zip(preds, y_test, trial_test_ids):
    trial_preds[tid].append(p)
    trial_true[tid] = y

final_preds = []
final_true = []

for tid in trial_preds:
    final_preds.append(np.bincount(trial_preds[tid]).argmax())
    final_true.append(trial_true[tid])

print("Trial accuracy:", accuracy_score(final_true, final_preds))
print("Confusion matrix:\n", confusion_matrix(final_true, final_preds))

Baseline Trial Accuracy: 0.4678107923497268
Confusion Matrix:
 [[6961 3457 2636]
 [2309 2349  771]
 [2188 1105 1648]]


In [ ]:
# ===== Save encoder after Temporal + Frequency SSL (Notebook-2) =====
save_path = "/content/drive/MyDrive/EEG_TSSL_DATA/encoder_cnn_transformer_tfssl.pt"

model.eval()  # optional but clean
torch.save(model.state_dict(), save_path)

print(f"[Notebook-2] Encoder saved to: {save_path}")

[Notebook-2] Encoder saved to: /content/drive/MyDrive/EEG_TSSL_DATA/encoder_cnn_transformer_tfssl.pt
